In [1]:
import os
import sys
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.model_selection import StratifiedKFold
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath("../.."))

from src.utils.target_encoding import target_encoding

In [2]:
# Configuration
ID = "026"
SEED = 42
N_SPLITS = 5
FEATURE_DIR = Path(f"../../artifacts/features/base/{ID}")

os.makedirs(FEATURE_DIR, exist_ok=True)

# === Chank処理の個数と処理する番目を確定 ===
CHUNK_SIZE = 8
CHUNK_N = 0  # 0~7

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pl.Config.set_tbl_rows(500)
pl.Config.set_tbl_cols(500)

polars.config.Config

### 1. 特徴量エンジニアリング (Chunkごとに保存)

In [3]:
# === Load Data ===
train = pl.read_csv("../../input/train.csv").drop("id")
test = pl.read_csv("../../input/test.csv").drop("id")
orig = pl.read_parquet("../../input/original.parquet")
orig = orig.with_columns(
    pl.when(pl.col("y") == "yes")
    .then(1)
    .when(pl.col("y") == "no")
    .then(0)
    .otherwise(None)
    .alias("y")
)

y_tr = train["y"].cast(pl.Int8)
y_orig = orig["y"].cast(pl.Int8)
y_merged = pl.concat([y_tr, y_orig], how="vertical")

train = train.drop("y")
orig = orig.drop("y")

CATS = [col for col in train.columns if train[col].dtype == pl.Utf8]
NUMS = [col for col in train.columns if train[col].dtype != pl.Utf8]

In [4]:
# === 全データを結合 ===
all_data = pl.concat([train, orig, test], how="vertical")

In [5]:
# === NUM → CAT ===
NUMS2CATS = [f"{c}2" for c in NUMS]
SIZES = {}

num2cat_exprs = [
    pl.col(c)
    .cast(pl.Utf8)
    .cast(pl.Categorical)
    .to_physical()
    .cast(pl.Int32)
    .alias(f"{c}2")
    for c in NUMS
]
cat_exprs = [
    pl.col(c).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(c) for c in CATS
]

all_data = all_data.with_columns(num2cat_exprs + cat_exprs)

SIZES = all_data.select(
    [pl.col(col).n_unique().alias(col) for col in CATS + NUMS2CATS]
).to_dicts()[0]

print(f"Created {len(NUMS2CATS)} new columns\n")
print(SIZES)

Created 7 new columns

{'job': 12, 'marital': 3, 'education': 4, 'default': 2, 'housing': 2, 'loan': 2, 'contact': 3, 'month': 12, 'poutcome': 4, 'age2': 78, 'balance2': 8590, 'day2': 31, 'duration2': 1824, 'campaign2': 52, 'pdays2': 628, 'previous2': 54}


In [6]:
# === 2Comboのペアを作成 ===
pairs = list(combinations(CATS + NUMS2CATS, 2))

combo_exprs = [
    (pl.col(c1) * SIZES[c2] + pl.col(c2)).alias(f"{c1}_{c2}") for c1, c2 in pairs
]

COMBO2 = [f"{c1}_{c2}" for c1, c2 in pairs]

all_data = all_data.with_columns(combo_exprs)

print(f"Created {len(combo_exprs)} new columns")

Created 120 new columns


In [7]:
# === 3Comboのペアを作成 ===
pairs = list(combinations(CATS + NUMS2CATS, 3))

combo_exprs = [
    ((pl.col(c1) * SIZES[c2] + pl.col(c2)) * SIZES[c3] + pl.col(c3)).alias(
        f"{c1}_{c2}_{c3}"
    )
    for c1, c2, c3 in pairs
]

COMBO3 = [f"{c1}_{c2}_{c3}" for c1, c2, c3 in pairs]

all_data = all_data.with_columns(combo_exprs)

print(f"Created {len(combo_exprs)} new columns")

Created 560 new columns


In [8]:
# === Target Encoding ===
tr_df = all_data[: len(train) + len(orig)].with_columns(y_merged.alias("target"))
test_df = all_data[len(train) + len(orig) :]

te_cols = NUMS2CATS + CATS + COMBO2 + COMBO3

n = CHUNK_SIZE
k, m = divmod(len(te_cols), n)
te_col_chunks = [
    te_cols[i * k + min(i, m) : (i + 1) * k + min(i + 1, m)] for i in range(n)
]
te_cols = te_col_chunks[CHUNK_N]

te_df = target_encoding(tr_df, test_df, cat_cols=te_cols)

print(f"Created {len(te_df.columns)} new columns")

0it [00:00, ?it/s]

  0%|          | 0/87 [00:00<?, ?it/s]

  0%|          | 0/87 [00:00<?, ?it/s]

  0%|          | 0/87 [00:00<?, ?it/s]

  0%|          | 0/87 [00:00<?, ?it/s]

  0%|          | 0/87 [00:00<?, ?it/s]

Created 87 new columns


In [9]:
# === Count Encoding ===
ce_cols = [c for c in all_data.columns if c not in NUMS2CATS]
k, m = divmod(len(ce_cols), n)
ce_col_chunks = [
    ce_cols[i * k + min(i, m) : (i + 1) * k + min(i + 1, m)] for i in range(n)
]
ce_cols = ce_col_chunks[CHUNK_N]

ce_dict = {f"{col}_ce": np.zeros(all_data.height) for col in ce_cols}
for col in tqdm(ce_cols):
    counts = all_data.group_by(col).agg(pl.len().alias(f"{col}_ce"))
    joined_df = all_data.join(counts, on=col, how="left")
    ce_dict[f"{col}_ce"] = joined_df[f"{col}_ce"]

ce_df = pl.DataFrame(ce_dict).with_columns(
    [pl.col(col).cast(pl.Float32) for col in ce_dict.keys()]
)

print(f"Created {len(ce_df.columns)} new columns")

  0%|          | 0/87 [00:00<?, ?it/s]

Created 87 new columns


In [10]:
# === 最初のChunkに数値データとidを付与 ===
if CHUNK_N == 0:
    all_data = pl.concat([all_data.select(NUMS), te_df, ce_df], how="horizontal")
    all_data = all_data.with_row_index("row_id")
else:
    all_data = pl.concat([te_df, ce_df], how="horizontal")

In [11]:
# === Downcast ===
INT32_MIN, INT32_MAX = -2_147_483_648, 2_147_483_647

all_data = all_data.with_columns(pl.col(pl.Float64).cast(pl.Float32))

# Int64で安全に落とせる列だけ選別
int64_cols = [c for c, dt in all_data.schema.items() if dt == pl.Int64]
safe_cols = []
for c in int64_cols:
    mn, mx = all_data[c].min(), all_data[c].max()
    if mn >= INT32_MIN and mx <= INT32_MAX:
        safe_cols.append(c)

# 安全な列だけ Int32 に
if safe_cols:
    all_data = all_data.with_columns(pl.col(safe_cols).cast(pl.Int32))


# === データを分割 ===
tr_df = all_data[: len(train) + len(orig)]
test_df = all_data[len(train) + len(orig) :]

# === targetを追加 ===
if CHUNK_N == 0:
    tr_df = tr_df.with_columns(y_merged.alias("target"))

In [12]:
# === 特徴量エンジニアリング後の情報 ===
tr_memory = sum(tr_df[col].to_numpy().nbytes for col in tr_df.columns) / 1024**3
test_memory = sum(test_df[col].to_numpy().nbytes for col in test_df.columns) / 1024**3

print("=== Shape & Memory ===")
print(f"Train Shape: {tr_df.shape}, Test Shape: {test_df.shape}")
print(f"Train Memory: {tr_memory:.2f} GB, Test Memory: {test_memory:.2f} GB\n")

dtype_counts = Counter([str(dt) for dt in tr_df.dtypes])

print("=== DTypes ===")
for dtype, cnt in dtype_counts.items():
    print(f"{dtype}: {cnt}")

=== Shape & Memory ===
Train Shape: (795211, 183), Test Shape: (250000, 182)
Train Memory: 0.54 GB, Test Memory: 0.17 GB

=== DTypes ===
UInt32: 1
Int32: 7
Float32: 174
Int8: 1


In [13]:
# === Save Chunk Data ===
tr_path = os.path.join(FEATURE_DIR, f"tr_df{ID}-chunk{CHUNK_N}.parquet")
test_path = os.path.join(FEATURE_DIR, f"test_df{ID}-chunk{CHUNK_N}.parquet")

tr_df.write_parquet(tr_path)
test_df.write_parquet(test_path)
print(f"\nFEATURE ENGINEERING CHNK_NO.{CHUNK_N} COMPLETE!")


FEATURE ENGINEERING CHNK_NO.0 COMPLETE!


### 2. Chunkをまとめる

In [4]:
# === Chunk Dataの統合 ===
train_list = []
test_list = []

for i in range(CHUNK_SIZE):
    train = pl.read_parquet(FEATURE_DIR / f"tr_df026-chunk{i}.parquet")
    train_list.append(train)
    test = pl.read_parquet(FEATURE_DIR / f"test_df026-chunk{i}.parquet")
    test_list.append(test)

tr_df = pl.concat([df for df in train_list], how="horizontal")
test_df = pl.concat([df for df in test_list], how="horizontal")

y = tr_df["target"]
tr_df = tr_df.drop("target")

all_data = pl.concat([tr_df, test_df], how="vertical")

In [5]:
# === 特徴量エンジニアリング後の情報 ===
tr_memory = sum(tr_df[col].to_numpy().nbytes for col in tr_df.columns) / 1024**3
test_memory = sum(test_df[col].to_numpy().nbytes for col in test_df.columns) / 1024**3

print("=== Shape & Memory ===")
print(f"Train Shape: {tr_df.shape}, Test Shape: {test_df.shape}")
print(f"Train Memory: {tr_memory:.2f} GB, Test Memory: {test_memory:.2f} GB\n")

dtype_counts = Counter([str(dt) for dt in tr_df.dtypes])

print("=== DTypes ===")
for dtype, cnt in dtype_counts.items():
    print(f"{dtype}: {cnt}")

=== Shape & Memory ===
Train Shape: (795211, 1400), Test Shape: (250000, 1400)
Train Memory: 4.15 GB, Test Memory: 1.30 GB

=== DTypes ===
UInt32: 1
Int32: 7
Float32: 1392


In [6]:
# === targetを追加 ===
tr_df = tr_df.with_columns(y.alias("target"))

In [7]:
# === Save Overall Data ===
tr_path = FEATURE_DIR / f"tr_df{ID}.parquet"
test_path = FEATURE_DIR / f"test_df{ID}.parquet"

tr_df.write_parquet(tr_path)
test_df.write_parquet(test_path)

### 3. Foldの列を作成

In [3]:
# === FOldの列を作成
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

tr_df = pl.read_parquet(FEATURE_DIR / f"tr_df{ID}.parquet")
y = tr_df["target"]

y_np = y.to_numpy()
X_dummy = np.arange(len(y_np))

fold_ids = np.zeros(len(y), dtype=int)

for fold_idx, (_, val_idx) in enumerate(tqdm(skf.split(X_dummy, y))):
    fold_ids[val_idx] = fold_idx

# fold列を追加
col_name = f"{N_SPLITS}fold-seed{SEED}"
tr_df = tr_df.with_columns(
    pl.Series(col_name, fold_ids).cast(pl.Int8)
)

# まとめて保存
out_path = FEATURE_DIR / f"tr_df{ID}-seed{SEED}.parquet"
tr_df.write_parquet(out_path)

0it [00:00, ?it/s]